# Exploring an indexed parquet shard

This notebook loads a parquet shard produced by `chess_corpus.index_games` and pokes at its shape.

It defaults to `data/sample/games/` (a 100-game sample of master-level OTB games carved from Lumbra's Gigabase OTB Elite). Point `SHARD_DIR` at `data/processed/games/` once you've run `make data` to ingest the full Lumbra corpus.

In [ ]:
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SHARD_DIR = ROOT / "data" / "sample" / "games"
shards = sorted(SHARD_DIR.glob("*.parquet"))
shards

## Schema and row count

In [ ]:
pf = pq.ParquetFile(shards[0])
print(f"rows: {pf.metadata.num_rows:,}")
print(f"row groups: {pf.metadata.num_row_groups}")
print()
print(pf.schema_arrow)

## A few rows as a DataFrame

In [ ]:
df = pq.read_table(shards[0]).to_pandas()
pd.set_option("display.max_colwidth", 80)
df.drop(columns=["pgn"]).head(20)

## The preserved PGN for one game

In [ ]:
print(df.iloc[0]["pgn"])

## Replay a game with python-chess

This is the same pattern any query script uses: take the `pgn` column, parse it, walk the mainline.

In [ ]:
import io

import chess
import chess.pgn

game = chess.pgn.read_game(io.StringIO(df.iloc[0]["pgn"]))
board = game.board()
for i, move in enumerate(game.mainline_moves(), start=1):
    san = board.san(move)
    board.push(move)
    on_b5 = board.piece_at(chess.B5)
    print(f"{i:>3}. {san:<8}  b5={on_b5.symbol() if on_b5 else '.'}")

## Quick group-by with DuckDB

DuckDB reads parquet shards directly with no import step. Handy for header-level slicing before you replay games.

In [ ]:
import duckdb

glob = str(SHARD_DIR / "*.parquet")
duckdb.sql(f"""
    SELECT eco,
           count(*) AS games,
           avg(white_elo)::INT AS avg_white_elo,
           avg(black_elo)::INT AS avg_black_elo
    FROM '{glob}'
    GROUP BY eco
    ORDER BY games DESC
""").df()